# Qwen legal: no-RAG vs official-source RAG

Notebook ini membandingkan adapter Qwen legal yang sama dalam dua kondisi:

1. `adapter_no_rag`: model menjawab tanpa kutipan hukum yang diambil saat inferensi.
2. `adapter_with_rag`: retriever mengambil potongan fixture sumber resmi berdasarkan identitas peraturan dan pasal, lalu model menjawab dengan instruksi grounding dan sitasi `[S1]`.

> Fixture ini hanya demonstrasi kecil. Ia bukan corpus hukum resmi lengkap. Verifikasi naskah, perubahan, status, dan currentness sebelum penggunaan produksi.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

repo_root = Path('/home/tamaniga34/notebooks/c5-legal')
runner = repo_root / 'notebooks' / 'qwen35_legal_rag_comparison.py'
run_root = Path('/home/tamaniga34/notebooks/qwen35_legal_rag_runs')
print('Runner:', runner)
print('Fixture:', repo_root / 'data/samples/official_source_rag_fixture.jsonl')

## Smoke run

Jalankan cell berikut untuk empat kasus singkat. Model dimuat satu kali dan digunakan untuk kedua kondisi.

In [ ]:
env = os.environ.copy()
env['QWEN_LEGAL_RAG_MODE'] = 'smoke'
result = subprocess.run(
    [sys.executable, str(runner)],
    cwd=repo_root,
    env=env,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError(f'Smoke run gagal dengan kode {result.returncode}')

In [ ]:
runs = sorted((run_root / 'smoke').glob('*'))
latest = runs[-1]
manifest = json.loads((latest / 'rag_comparison_manifest.json').read_text())
print('Latest run:', latest)
for label, summary in manifest['automatic_diagnostics'].items():
    print(label, summary)
print('Report:', manifest['outputs']['comparison_report_md'])
print('Human review:', manifest['outputs']['human_review_queue_csv'])

## Inspect per-case answers

Baca output model berdampingan. Untuk penilaian legal, isi kolom reviewer pada `human_review_queue.csv`.

In [ ]:
outputs_path = latest / 'comparison_outputs.jsonl'
for line in outputs_path.read_text(encoding='utf-8').splitlines():
    row = json.loads(line)
    print('\n###', row['case_id'])
    print('Q:', row['prompt'])
    print('No RAG:', row['adapter_no_rag']['answer'])
    print('Dengan RAG:', row['adapter_with_rag']['answer'])
    print('Scores:', row['adapter_no_rag']['answer_screening_score'], row['adapter_with_rag']['answer_screening_score'], row['adapter_with_rag']['rag_grounding_score'])

## Full run

Set `QWEN_LEGAL_RAG_MODE` menjadi `full` pada cell berikut jika ingin menjalankan seluruh kasus dalam file test. Smoke memakai empat kasus pertama; full memakai seluruh enam kasus pada fixture saat ini.

In [ ]:
env = os.environ.copy()
env['QWEN_LEGAL_RAG_MODE'] = 'full'
result = subprocess.run(
    [sys.executable, str(runner)],
    cwd=repo_root,
    env=env,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError(f'Full run gagal dengan kode {result.returncode}')